In [6]:
import re

class PorterStemmer:
    def __init__(self):
        self.vowels = "aeiou"

    def is_consonant(self, word, i):
        if word[i] in self.vowels:
            return False
        if word[i] == 'y':
            return i == 0 or not self.is_consonant(word, i - 1)
        return True

    def measure(self, word):
        """Count VC sequences"""
        m = 0
        i = 0
        length = len(word)

        while i < length:
            if not self.is_consonant(word, i):
                break
            i += 1

        while i < length:
            while i < length and not self.is_consonant(word, i):
                i += 1
            while i < length and self.is_consonant(word, i):
                i += 1
                if i < length:
                    m += 1
        return m

    def contains_vowel(self, word):
        return any(not self.is_consonant(word, i) for i in range(len(word)))

    def ends_double_consonant(self, word):
        return len(word) >= 2 and word[-1] == word[-2] and self.is_consonant(word, len(word)-1)

    def cvc(self, word):
        if len(word) < 3:
            return False
        if (self.is_consonant(word, -1) and
            not self.is_consonant(word, -2) and
            self.is_consonant(word, -3)):
            if word[-1] not in "wxy":
                return True
        return False

    # --- Step 1a ---
    def step1a(self, word):
        if word.endswith("sses"):
            return word[:-2]
        elif word.endswith("ies"):
            return word[:-2]
        elif word.endswith("ss"):
            return word
        elif word.endswith("s"):
            return word[:-1]
        return word

    # --- Step 1b ---
    def step1b(self, word):
        if word.endswith("eed"):
            stem = word[:-3]
            if self.measure(stem) > 0:
                return stem + "ee"
        elif word.endswith("ed"):
            stem = word[:-2]
            if self.contains_vowel(stem):
                return self.step1b_helper(stem)
        elif word.endswith("ing"):
            stem = word[:-3]
            if self.contains_vowel(stem):
                return self.step1b_helper(stem)
        return word

    def step1b_helper(self, word):
        if word.endswith(("at", "bl", "iz")):
            return word + "e"
        elif self.ends_double_consonant(word) and word[-1] not in "lsz":
            return word[:-1]
        elif self.measure(word) == 1 and self.cvc(word):
            return word + "e"
        return word

    # --- Step 1c ---
    def step1c(self, word):
        if word.endswith("y"):
            stem = word[:-1]
            if self.contains_vowel(stem):
                return stem + "i"
        return word

    # --- Step 2 ---
    def step2(self, word):
        rules = {
            "ational": "ate", "tional": "tion",
            "enci": "ence", "anci": "ance",
            "izer": "ize", "abli": "able",
            "alli": "al", "entli": "ent",
            "eli": "e", "ousli": "ous",
            "ization": "ize", "ation": "ate",
            "ator": "ate", "alism": "al",
            "iveness": "ive", "fulness": "ful",
            "ousness": "ous", "aliti": "al",
            "iviti": "ive", "biliti": "ble"
        }

        for key in rules:
            if word.endswith(key):
                stem = word[:-len(key)]
                if self.measure(stem) > 0:
                    return stem + rules[key]
        return word

    # --- Step 3 ---
    def step3(self, word):
        rules = {
            "icate": "ic", "ative": "",
            "alize": "al", "iciti": "ic",
            "ical": "ic", "ful": "",
            "ness": ""
        }

        for key in rules:
            if word.endswith(key):
                stem = word[:-len(key)]
                if self.measure(stem) > 0:
                    return stem + rules[key]
        return word

    # --- Step 4 ---
    def step4(self, word):
        suffixes = [
            "al", "ance", "ence", "er", "ic",
            "able", "ible", "ant", "ement",
            "ment", "ent", "ion", "ou",
            "ism", "ate", "iti", "ous",
            "ive", "ize"
        ]

        for s in suffixes:
            if word.endswith(s):
                stem = word[:-len(s)]
                if self.measure(stem) > 1:
                    if s == "ion":
                        if stem.endswith("s") or stem.endswith("t"):
                            return stem
                    else:
                        return stem
        return word

    # --- Step 5 ---
    def step5(self, word):
        if word.endswith("e"):
            stem = word[:-1]
            if self.measure(stem) > 1:
                return stem
            if self.measure(stem) == 1 and not self.cvc(stem):
                return stem

        if self.measure(word) > 1 and self.ends_double_consonant(word) and word.endswith("l"):
            return word[:-1]

        return word

    def stem(self, word):
        word = word.lower()
        word = self.step1a(word)
        word = self.step1b(word)
        word = self.step1c(word)
        word = self.step2(word)
        word = self.step3(word)
        word = self.step4(word)
        word = self.step5(word)
        return word


# --- Testing ---
if __name__ == "__main__":
    stemmer = PorterStemmer()
    words = ["computers", "computing", "computed", "relational", "happiness","Running","playing"]

    for w in words:
        print(f"{w} -> {stemmer.stem(w)}")

computers -> comput
computing -> comput
computed -> comput
relational -> relation
happiness -> happi
Running -> run
playing -> plai
